# 03 · Spark SQL

**Objetivos**
- Registrar DataFrames como vistas y consultarlos con SQL estándar (Armbrust et al., 2015).
- Usar CTEs, funciones de ventana y `CASE WHEN`.
- Crear y usar una UDF.
- Comparar la API de DataFrames con SQL (mismo plan de ejecución).

> Duración estimada: 25 min

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Fuera de Docker -> local[*]. Dentro del docker-compose -> spark://spark-master:7077
MASTER = os.environ.get("SPARK_MASTER", "local[*]")

builder = (SparkSession.builder
           .appName("03_SparkSQL")
           .master(MASTER)
           .config("spark.sql.shuffle.partitions", "8"))   # pocos datos -> pocas particiones
if os.environ.get("SPARK_DRIVER_HOST"):
    builder = builder.config("spark.driver.host", os.environ["SPARK_DRIVER_HOST"])

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

DATA = os.path.abspath("../data")   # carpeta de datos del repositorio
print("Spark", spark.version, "| master:", sc.master)
print("Spark UI:", sc.uiWebUrl)

## 1. Registrar vistas temporales

In [ ]:
spark.read.csv(f"{DATA}/ventas.csv", header=True, inferSchema=True).createOrReplaceTempView("ventas")
spark.read.csv(f"{DATA}/clientes.csv", header=True, inferSchema=True).createOrReplaceTempView("clientes")
spark.read.json(f"{DATA}/productos.json").createOrReplaceTempView("productos")

spark.sql("SHOW TABLES").show()

## 2. Consultas básicas

In [ ]:
spark.sql("""
    SELECT canal,
           COUNT(*)                                          AS num_ventas,
           ROUND(SUM(cantidad * precio_unitario * (1 - descuento))) AS ingresos
    FROM ventas
    WHERE cantidad > 0
    GROUP BY canal
    ORDER BY ingresos DESC
""").show()

## 3. Vista limpia y enriquecida (CTE + JOIN)

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW ventas_full AS
WITH limpias AS (
    SELECT *,
           cantidad * precio_unitario * (1 - descuento) AS total,
           MONTH(fecha) AS mes
    FROM ventas
    WHERE cantidad IS NOT NULL AND cantidad > 0
)
SELECT l.*, p.producto, p.categoria, c.nombre, c.ciudad, c.region, c.segmento, c.edad
FROM limpias l
LEFT JOIN productos p ON l.id_producto = p.id_producto
LEFT JOIN clientes  c ON l.id_cliente  = c.id_cliente
""")
spark.sql("SELECT id_venta, fecha, producto, categoria, ciudad, total FROM ventas_full LIMIT 5").show()

## 4. `CASE WHEN`: rangos de edad

In [ ]:
spark.sql("""
SELECT CASE WHEN edad < 25 THEN '18-24'
            WHEN edad < 35 THEN '25-34'
            WHEN edad < 50 THEN '35-49'
            ELSE '50+' END            AS rango_edad,
       COUNT(*)                        AS ventas,
       ROUND(AVG(total))               AS ticket_promedio
FROM ventas_full
GROUP BY 1
ORDER BY 1
""").show()

## 5. Funciones de ventana
Top 3 productos más vendidos **por categoría** usando `RANK()`.

In [ ]:
spark.sql("""
WITH por_producto AS (
    SELECT categoria, producto, SUM(cantidad) AS unidades
    FROM ventas_full
    GROUP BY categoria, producto
)
SELECT * FROM (
    SELECT *, RANK() OVER (PARTITION BY categoria ORDER BY unidades DESC) AS ranking
    FROM por_producto
) WHERE ranking <= 3
ORDER BY categoria, ranking
""").show(20, truncate=False)

Ingresos mensuales y **acumulado** del año:

In [ ]:
spark.sql("""
SELECT mes,
       ROUND(SUM(total) / 1e6, 1)                                AS ingresos_mill,
       ROUND(SUM(SUM(total)) OVER (ORDER BY mes) / 1e6, 1)       AS acumulado_mill
FROM ventas_full
GROUP BY mes
ORDER BY mes
""").show()

## 6. UDF (User Defined Function)
> Las UDF de Python son más lentas que las funciones nativas, porque los datos deben transferirse de la JVM al intérprete de Python. Su uso se reserva para los casos en que no existe una función integrada (Damji et al., 2020).

In [ ]:
from pyspark.sql.types import StringType

def clasificar_ticket(total):
    if total is None:
        return None
    if total >= 1_000_000:
        return "Alto"
    if total >= 200_000:
        return "Medio"
    return "Bajo"

spark.udf.register("clasificar_ticket", clasificar_ticket, StringType())

spark.sql("""
SELECT clasificar_ticket(total) AS tipo_ticket, COUNT(*) AS ventas
FROM ventas_full GROUP BY 1 ORDER BY ventas DESC
""").show()

## 7. SQL y DataFrames generan el mismo plan

In [ ]:
q_sql = spark.sql("SELECT categoria, SUM(total) AS t FROM ventas_full GROUP BY categoria")
q_df  = spark.table("ventas_full").groupBy("categoria").agg(F.sum("total").alias("t"))

q_sql.explain()
q_df.explain()

## Ejercicios propuestos (en SQL)

1. Determinar la **ciudad** con el mayor ticket promedio.
2. Identificar, para cada **región**, la categoría con más ingresos (sugerencia: `ROW_NUMBER()`).
3. Calcular el porcentaje de ventas realizadas con descuento.
4. Calcular el crecimiento porcentual de los ingresos de cada mes respecto al anterior (sugerencia: `LAG()`).

In [ ]:
# Ejercicio 1
# spark.sql("""
#     SELECT ...
# """).show()

In [ ]:
# Ejercicio 2

In [ ]:
# Ejercicio 3

In [ ]:
# Ejercicio 4

In [ ]:
spark.stop()

## Referencias

Armbrust, M., Xin, R. S., Lian, C., Huai, Y., Liu, D., Bradley, J. K., Meng, X., Kaftan, T., Franklin, M. J., Ghodsi, A., & Zaharia, M. (2015). Spark SQL: Relational data processing in Spark. En *Proceedings of the 2015 ACM SIGMOD International Conference on Management of Data* (pp. 1383–1394). ACM. https://doi.org/10.1145/2723372.2742797

Damji, J. S., Wenig, B., Das, T., & Lee, D. (2020). *Learning Spark: Lightning-fast data analytics* (2.ª ed.). O'Reilly Media.